# Aralin 18 (pagsunod): Mga Resibo na nagpapatunay na isang *Tao* ang Nagbigay-pahintulot sa Aksyon

Pinatutunayan ng aralin kung ano ang ginawa ng **ahente** at kung ano ang napagpasyahan ng **pinto**. Idinadagdag ng talaang ito ang nawawalang kalahati: patunay na isang **pangalanang tao** ang nag-apruba sa **eksaktong** aksyon — isang hiwalay, hawak ng tao na pirma sa kabuuang kanonikong aksyon, na-verify nang offline.

Parehong gamit dito ng mga artifact ang **kaparehong hugis ng sobre gaya ng mga resibo sa aralin**: isang patag na nilalaman na may `type` field, pinirmahan gamit ang Ed25519 sa SHA-256 ng mga byte ng kanonikong nilalaman, na may nakalakip na structured na `signature` na object (na hindi kasama sa pinirmang mga byte). Ang resibo ng pag-apruba ay isang bagong `type` (`human.approval.v1`) kasabay ng uri ng aksyon, kaya isang `verify_chain` lang ang sumasaklaw sa parehong uri ng artifact gamit ang parehong daang pangkodigo na ginamit mo sa pangunahing talaan. Ito rin ang hugis ng landas ng co-signature sa Internet-Draft na sinusunod ng aralin (draft-farley-acta-signed-receipts).

Isang sinadyang pag-upgrade kumpara sa demo verifier sa pangunahing talaan: dito, ang verifier ay nireresolba ang `signature.key_id` laban sa isang **nakapirming rehistro ng susi** sa halip na magtiwala sa pampublikong susi na dala sa loob ng resibo. Ito ang posturang pang-produksyon na inirerekomenda ng sariling checklist ng aralin ("ipublish ang pampublikong susi sa pagbibigay-husay"), at ito ang dahilan kung bakit ang pamemeke ay tinatanggihan sa halip na papasa bilang isang bypass gamit ang sariling susi.

Ang tuntunin na itinuturo ng talaang ito: **ang pinirmang pag-apruba ay hindi awtoridad nang mag-isa.** Ang awtoridad ay umiiral lamang kung ang resibo ng pag-apruba at resibo ng aksyon ay nananatiling naka-bind sa iisang kanonikong aksyon sa oras ng pagpapatupad, sa ilalim ng isang bersyon ng polisiya, susi, at petsa ng pag-expire na kasalukuyan pa rin, at isang pag-apruba na hindi pa nagagamit. Bawat pagkabigo ay tumatanggi gamit ang **natatanging dahilan**, kaya malalaman mong *lipas na ang awtoridad* mula sa *nagbago ang naipatupad na aksyon*.


In [1]:
# These are already the Lesson 18 dependencies — no new packages.
# %pip install pynacl jcs
import base64, copy, hashlib
from jcs import canonicalize                      # RFC 8785 canonical JSON
from nacl.signing import SigningKey, VerifyKey
# CryptoError is the common base of BadSignatureError AND the ValueError pynacl
# raises for a wrong-length signature — catch the base so verification fails
# closed on ANY bad signature, not just the forged-but-correct-length one.
from nacl.exceptions import CryptoError

# Same helpers as the main notebook.
def b64url_nopad(data: bytes) -> str:
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    return base64.urlsafe_b64decode(s + "=" * ((4 - len(s) % 4) % 4))

def sha256_canonical(obj) -> str:
    """SHA-256 of an object's JCS-canonical JSON form (same helper as the lesson)."""
    return f"sha256:{hashlib.sha256(canonicalize(obj)).hexdigest()}"

## Ang eksaktong aksyon

Ang yunit ng pag-apruba ay ang **canonical action object** — hindi isang malabong label tulad ng "aprubahan ang refund," kundi ang eksakto, ganap na tinukoy na aksyon. Ang paglagda sa buong bagay (at pagkuha ng digest mula rito) ang nagpapahintulot sa atin na patunayan sa hinaharap na ang tao ay nag-apruba *dito* at wala nang iba pa.


In [2]:
action = {
    "action_type": "refund.issue",
    "params": {"order_id": "A-1029", "amount_usd": 4200, "to": "acct_88"},
    "policy_id": "refunds-v3",
}
print("action digest:", sha256_canonical(action))

action digest: sha256:fba342ad8447b491a089d7a09d4ac58f1a835c504e58f8d832db04f65bb62a25


## Isang sobre, dalawang awtoridad

Ang bawat resibo ay ang sobre ng aralin: isang patag na payload na may `type` na patlang, kasama ang isang `signature` na bagay (`alg`, `sig`, `key_id`) na **hindi** bahagi ng pinirmahang mga bytes. Ang `verify_envelope` ay ang pinagsaluhang estruktura + tseke sa pirma para sa parehong uri ng resibo; kung saang **nakapirmang rehistro ng susi** nito ni-reresolba ang `signature.key_id` ang nagbibigay-daan upang mapanatiling hiwalay ang mga awtoridad:

- **approval receipt** (`human.approval.v1`) — tinukoy ang approver, ang buong kanonikal na aksyon **at ang digest nito**, `policy_version`, oras ng isyu + pag-expire. Ang one-time consumption ay sinusubaybayan sa antas ng chain.
- **action receipt** (`agent.action.v1`) — pagkakakilanlan ng ahente, `run_id`, ang parehong kanonikal na aksyon **digest**, resulta ng pagpapatupad + timestamp, at `parent_approval_ref`: ang `receipt_hash` ng aprubasyon, ang parehong konbensyon tulad ng `previous_receipt_hash` sa chain ng aralin.

Ang pinagsaluhang `action_digest` na patlang ang pinag-uugnayan na kinakailangan. Ang `key_id` ay nasa loob ng signature na bagay bilang isang palatandaan sa paghahanap lamang: ang pag-turo nito sa ibang naka-pin na susi ay nagpapabigo sa tseke ng pirma, kaya wala itong ipinagkakaloob.


In [3]:
# ---- pinned key registries: SEPARATE authorities, one envelope shape ----------
# Published out of band (the lesson checklist's JWK-Set pattern); the verifier
# NEVER trusts a key carried inside a receipt.
approver_sk = SigningKey.generate()
agent_sk    = SigningKey.generate()
APPROVER_KEYS = {"approver-key-1": b64url_nopad(bytes(approver_sk.verify_key))}
AGENT_KEYS    = {"agent-key-1":    b64url_nopad(bytes(agent_sk.verify_key))}

# The policy the approval is granted under. If this moves after approval, the
# approval is STALE even though its signature still verifies.
CURRENT_POLICY = {"policy_version": "refunds-v3"}

def sign_receipt(payload: dict, sk: SigningKey, key_id: str) -> dict:
    """Same signing pipeline as the lesson: Ed25519 over the SHA-256 of the
    canonical payload; the signature object is NOT part of the signed bytes."""
    message_hash = hashlib.sha256(canonicalize(payload)).digest()
    return {
        **payload,
        "signature": {"alg": "EdDSA", "sig": b64url_nopad(sk.sign(message_hash).signature), "key_id": key_id},
    }

def verify_envelope(receipt, expected_type: str, trusted_keys: dict):
    """The SHARED verifier contract for any receipt kind; the caller picks which
    pinned registry (authority) resolves key_id. Fails closed on ANY
    attacker-shaped input: malformed is a refusal, never a crash."""
    if not isinstance(receipt, dict) or not isinstance(receipt.get("signature"), dict):
        return (False, "receipt malformed (not an object with a signature object)")
    sig_obj = receipt["signature"]
    if sig_obj.get("alg") != "EdDSA":
        return (False, "unsupported signature alg")
    if receipt.get("type") != expected_type:
        return (False, f"wrong receipt type (expected {expected_type})")
    # Key freshness is part of authority: a key_id rotated out of the pinned
    # registry confers nothing, even with a valid signature.
    pub = trusted_keys.get(sig_obj.get("key_id"))
    if pub is None:
        return (False, f"stale authority: key_id {sig_obj.get('key_id')!r} is not in the pinned registry (unknown or rotated out)")
    # Reconstruct the signed bytes exactly as the lesson does: everything except
    # the signature object, canonicalized, hashed.
    payload = {k: v for k, v in receipt.items() if k != "signature"}
    try:
        message_hash = hashlib.sha256(canonicalize(payload)).digest()
        VerifyKey(b64url_decode(pub)).verify(message_hash, b64url_decode(sig_obj.get("sig") or ""))
    except (CryptoError, TypeError, ValueError, base64.binascii.Error):
        return (False, "signature invalid (forged, tampered, or malformed)")
    return (True, "envelope ok")

def human_approval(action, approver_id, approved_at, sk=approver_sk,
                   key_id="approver-key-1", policy_version=None, expires_at=None):
    # deepcopy: the receipt must be an immutable record of what was approved —
    # a live reference would let a later mutation of `action` silently change the
    # signed payload. Digest the SNAPSHOT so the two can never diverge.
    approved_action = copy.deepcopy(action)
    payload = {
        "type": "human.approval.v1",
        "approver_id": approver_id,
        "action": approved_action,                       # the FULL canonical action
        "action_digest": sha256_canonical(approved_action),  # the join field
        "policy_version": policy_version or CURRENT_POLICY["policy_version"],
        "approved_at": approved_at,                      # ISO-8601 Zulu, like the lesson
        "expires_at": expires_at or approved_at[:11] + "23:59:59Z",
    }
    return sign_receipt(payload, sk, key_id)

In [4]:
approval = human_approval(action, "alice@ops (WebAuthn)", "2026-07-08T15:04:05Z",
                          expires_at="2026-07-08T15:19:05Z")
print(verify_envelope(approval, "human.approval.v1", APPROVER_KEYS))
print("binds digest:", approval["action_digest"][:23], "…  under", approval["policy_version"])

(True, 'envelope ok')
binds digest: sha256:fba342ad8447b491 …  under refunds-v3


## `verify_chain`: kung saan talagang pinagpapasyahan ang binding

Ang `verify_chain` ay **hindi** isang maginhawang pambalot sa dalawang tseke ng pirma. Ito ang iisang lugar kung saan ang ibinahaging canonikal na `action_digest`, ang policy/key/expiry **kasariwaan** ng pag-apruba, at ang **pagkonsumo nang isang beses** ng pag-apruba ay sabay-sabay na sinusuri, laban sa aksyon na isinasagawa *ngayon mismo*.

Bawat pagkabigo ay tumatanggi na may isang **natatanging dahilan**, kaya't ang nagbabasa ng pagtanggi ay maaaring malaman kung ang awtoridad ay luma na (nagbago ang polisiya, na-rotate ang susi, nag-expire ang pag-apruba, nagamit na ang pag-apruba) o ang isinasagawang aksyon ay nagbago mula sa ilalim ng isang pa rin valid na pag-apruba (pagpapalit ng digest).


In [5]:
def receipt_hash(receipt: dict) -> str:
    """Content-derived id of a COMPLETE receipt (including its signature) —
    the same convention as previous_receipt_hash in the lesson's chain."""
    return sha256_canonical(receipt)

def agent_receipt(action, approval, executed_at, sk=agent_sk, key_id="agent-key-1"):
    executed_action = copy.deepcopy(action)    # snapshot, same reason as the approval
    payload = {
        "type": "agent.action.v1",
        "agent_id": "agent:refunds-bot",
        "run_id": "run-0001",
        "action": executed_action,
        "action_digest": sha256_canonical(executed_action),  # same join field
        "parent_approval_ref": receipt_hash(approval),
        "outcome": "performed",
        "executed_at": executed_at,
    }
    return sign_receipt(payload, sk, key_id)

_consumed = set()

def verify_chain(action_being_executed, approval, agent_rcpt, now: str):
    """One code path covers both receipt kinds (same envelope), then checks the
    things that only make sense TOGETHER: shared digest, freshness, consumption.
    `now` is an ISO-8601 Zulu timestamp; Zulu strings compare correctly as strings."""
    # 1. Shared envelope contract, separate authorities.
    ok, why = verify_envelope(approval, "human.approval.v1", APPROVER_KEYS)
    if not ok: return (False, f"approval: {why}")
    ok, why = verify_envelope(agent_rcpt, "agent.action.v1", AGENT_KEYS)
    if not ok: return (False, f"agent receipt: {why}")

    # 2. The join: BOTH receipts must bind the digest of the action being executed
    #    right now. A valid approval for a DIFFERENT action is substitution, and it
    #    gets its own reason — this is "the executed action changed".
    executing_digest = sha256_canonical(action_being_executed)
    if approval.get("action_digest") != executing_digest or approval.get("action") != action_being_executed:
        return (False, "digest substitution: the approval binds a different canonical action than the one being executed")
    if agent_rcpt.get("action_digest") != executing_digest or agent_rcpt.get("action") != action_being_executed:
        return (False, "digest substitution: the agent receipt binds a different canonical action than the one being executed")
    if agent_rcpt.get("parent_approval_ref") != receipt_hash(approval):
        return (False, "agent receipt is not bound to this approval")

    # 3. Freshness: a valid signature over stale authority is still a refusal —
    #    each staleness gets its own reason, distinct from substitution above.
    if approval.get("policy_version") != CURRENT_POLICY["policy_version"]:
        return (False, f"stale authority: approved under policy {approval.get('policy_version')!r}, current is {CURRENT_POLICY['policy_version']!r}")
    expires = approval.get("expires_at")
    if not isinstance(expires, str) or not expires or now >= expires:
        return (False, "stale authority: approval expired before execution")

    # 4. One-time consumption: an approval authorizes ONE execution.
    ref = receipt_hash(approval)
    if ref in _consumed:
        return (False, "approval already consumed (replay refused)")
    _consumed.add(ref)
    return (True, f"approved by {approval['approver_id']}, executed by {agent_rcpt['agent_id']}")

def execute(action, approval, agent_rcpt, now):
    ok, why = verify_chain(action, approval, agent_rcpt, now)
    return (ok, "executed" if ok else why)

receipt = agent_receipt(action, approval, "2026-07-08T15:04:06Z")
print(execute(action, approval, receipt, now="2026-07-08T15:04:07Z"))

(True, 'executed')


## Ano ang nahuhuli ng binding

Bawat kaso sa ibaba ay nabibigo nang **sarado** na may **ibang dahilan**. Ang unang bloke ay ang klasikong set (pandaraya, nalitong kinatawan, replay, pamemeke sa alinmang awtoridad, sira o maling input). Ang ikalawang bloke ay ang pares na ginagawa ang property na totoo sa halip na ipinahayag lamang:

- **lipas na awtoridad** — ang pirma ay valid pa, pero ang bersyon ng polisiya ay lumipat na, ang susi ng tagapahintulot ay pinalitan mula sa nakatapat na rehistro, o ang pag-apruba ay nag-expire bago ang pagsasagawa;
- **pagpapalit ng digest** — isang wastong pinirmahang resibo ng aksyon na ang `parent_approval_ref` ay tumutukoy sa isang *tunay* na pag-apruba, pero ang canonical na digest ng aksyon ng pag-aprubang iyon ay hindi tumutugma sa aktwal na isinasagawang aksyon.


In [6]:
NOW = "2026-07-08T15:05:00Z"

# 1. tamper: change the amount after approval — the executed action changed.
tampered = {**action, "params": {**action["params"], "amount_usd": 9900}}
print("tamper              ->", verify_chain(tampered, approval, agent_receipt(tampered, approval, NOW), NOW))

# 2. confused deputy: valid approval for action A, presented to execute action B.
action_b = {**action, "action_type": "wire.send"}
print("confused-deputy     ->", verify_chain(action_b, approval, agent_receipt(action_b, approval, NOW), NOW))

# 3. replay: the approval was consumed by the successful execution above.
print("replay              ->", execute(action, approval, agent_receipt(action, approval, NOW), NOW))

# 4. forged approval: attacker signs with their own key but claims a pinned key_id.
mallory_sk = SigningKey.generate()
forged = human_approval(action, "mallory", NOW, sk=mallory_sk)
print("forged-approval     ->", verify_chain(action, forged, agent_receipt(action, forged, NOW), NOW))

# A fresh, un-consumed approval so the agent-side cases fail on their OWN check.
fresh = human_approval(action, "alice@ops (WebAuthn)", NOW, expires_at="2026-07-08T15:20:00Z")

# 5. self-minted agent receipt: attacker's own agent key, refused by the pinned registry.
mallory_agent = agent_receipt(action, fresh, NOW, sk=SigningKey.generate())
print("self-minted-agent   ->", verify_chain(action, fresh, mallory_agent, NOW))

# 6. wrong-action agent receipt: real agent key, but the receipt binds a different action.
wrong_action = {**action, "params": {**action["params"], "amount_usd": 9900}}
print("wrong-action-agent  ->", verify_chain(action, fresh, agent_receipt(wrong_action, fresh, NOW), NOW))

# 7. malformed input: structurally broken receipts refuse cleanly, they never crash.
print("malformed-approval  ->", verify_chain(action, {"type": "human.approval.v1"}, agent_receipt(action, fresh, NOW), NOW))
print("malformed-agent     ->", verify_chain(action, fresh, {"nope": "not a receipt"}, NOW))

# 8. wrong-length signature: valid base64, not 64 bytes — refused, not crashed.
badlen = {**fresh, "signature": {**fresh["signature"], "sig": "AAAA"}}
print("wrong-len-sig       ->", verify_chain(action, badlen, agent_receipt(action, fresh, NOW), NOW))

# 9. non-object receipt: a list refuses cleanly instead of raising AttributeError.
print("nonobject-receipt   ->", verify_chain(action, [1, 2], agent_receipt(action, fresh, NOW), NOW))

print()
print("--- the two negative controls that make the property real ---")

# 10. STALE POLICY: signature still valid, but policy moved between approval and
#     execution. Authority is decided at execution time, not signing time.
CURRENT_POLICY["policy_version"] = "refunds-v4"
print("stale-policy        ->", verify_chain(action, fresh, agent_receipt(action, fresh, NOW), NOW))
CURRENT_POLICY["policy_version"] = "refunds-v3"   # restore for the cases below

# 11. STALE KEY: the approver key is rotated out of the pinned registry after
#     signing. The signature bytes still verify against the old key — but the old
#     key no longer confers authority.
rotated_out = APPROVER_KEYS.pop("approver-key-1")
print("stale-key           ->", verify_chain(action, fresh, agent_receipt(action, fresh, NOW), NOW))
APPROVER_KEYS["approver-key-1"] = rotated_out     # restore

# 12. EXPIRED: approval was valid when signed, but execution came too late.
expired = human_approval(action, "alice@ops (WebAuthn)", "2026-07-08T14:00:00Z",
                         expires_at="2026-07-08T14:01:00Z")
print("expired-approval    ->", verify_chain(action, expired, agent_receipt(action, expired, NOW), NOW))

# 13. DIGEST SUBSTITUTION: a validly signed agent receipt whose parent_approval_ref
#     points at a REAL approval — but that approval binds action B, and the agent
#     is executing action A. Distinct reason from every staleness above.
approval_b = human_approval(action_b, "alice@ops (WebAuthn)", NOW, expires_at="2026-07-08T15:20:00Z")
substituted = agent_receipt(action, approval_b, NOW)   # executing `action`, ref -> approval of action_b
print("digest-substitution ->", verify_chain(action, approval_b, substituted, NOW))

tamper              -> (False, 'digest substitution: the approval binds a different canonical action than the one being executed')
confused-deputy     -> (False, 'digest substitution: the approval binds a different canonical action than the one being executed')
replay              -> (False, 'approval already consumed (replay refused)')
forged-approval     -> (False, 'approval: signature invalid (forged, tampered, or malformed)')
self-minted-agent   -> (False, 'agent receipt: signature invalid (forged, tampered, or malformed)')
wrong-action-agent  -> (False, 'digest substitution: the agent receipt binds a different canonical action than the one being executed')
malformed-approval  -> (False, 'approval: receipt malformed (not an object with a signature object)')
malformed-agent     -> (False, 'agent receipt: receipt malformed (not an object with a signature object)')
wrong-len-sig       -> (False, 'approval: signature invalid (forged, tampered, or malformed)')
nonobject-receipt   -> (Fa

## Ano ang pinatutunayan nito — at hindi nito pinatutunayan

**Pinatutunayan:** isang pinangalang tao ang nag-apruba *ng eksaktong canonical na aksyon na ito* (buong aksyon + digest, nilagdaan gamit ang isang susi na nalutas mula sa isang pinned na rehistro), at ang ahente ay nagsagawa *ng eksaktong aprubadong aksyon na iyon* (kaparehong digest, resibo na naka-link sa apruba sa pamamagitan ng `receipt_hash`, sariling panuntunan ng aralin sa chain) — habang ang bersyon ng polisiya, susi, at pag-expire ng apruba ay kasalukuyan pa, isang beses lamang. Kung may magbago sa alinmang panig, ang chain ay magfa-fail ng closed, at ang dahilan ng pagtanggi ay magsasabi sa iyo **alin** sa mga katangiang iyon ang nasira: luma na ang awtoridad kumpara sa isang nabagong aksyon.

**Hindi pinatutunayan:** na ipinakita ng UI ng apruba sa tao ang eksaktong kanilang inisip na kanilang pipirmahan (WYSIWYS ay sarili nitong problema), na ang susi ay hindi pinilit o ninakaw bago ang pag-ikot, o na ang mga downstream na epekto ay tumugma sa aksyon. Ang pagkakalagda ay ≠ awtorisado: ang wastong lagda sa ibabaw ng luma nang polisiya, isang pinaikot na susi, isang expiro na bintana, o ibang digest ay walang naipapasa dito.

Ang dalawang uri ng resibo ay gumagamit ng parehong sobre ng aralin at isang `verify_chain` na code path nang sadyang ganito: ang binding na ginawa mo para sa mga resibo ng aksyon sa pangunahing notebook ay parehong code na sumusuri sa apruba ng tao. Isang kontrata ng tagasuri, magkahiwalay na pinned na awtoridad, pinagsama ng canonical na digest ng aksyon at wala nang iba pa.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Pagtatanggi**:
Ang dokumentong ito ay isinalin gamit ang serbisyo ng AI translation na [Co-op Translator](https://github.com/Azure/co-op-translator). Bagama't nagsusumikap kami para sa katumpakan, pakatandaan na ang awtomatikong pagsasalin ay maaaring maglaman ng mga pagkakamali o hindi pagkakatugma. Ang orihinal na dokumento sa orihinal nitong wika ang dapat ituring na pangunahing sanggunian. Para sa mahahalagang impormasyon, inirerekomenda ang propesyonal na pagsasalin ng tao. Hindi kami mananagot sa anumang maling pagkakaintindi o maling interpretasyon na nagmula sa paggamit ng pagsasaling ito.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
